In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

df = spark.read.csv("s3://aiml-final/ProjectTrainingData.csv", header=True)
df.show(5)
df.count()


Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1764391691638_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+-----+--------+----+----------+--------+-----------+-------------+--------+----------+------------+---------+---------+------------+-----------+----------------+-----+---+---+----+---+---+------+---+
|                  id|click|    hour|  C1|banner_pos| site_id|site_domain|site_category|  app_id|app_domain|app_category|device_id|device_ip|device_model|device_type|device_conn_type|  C14|C15|C16| C17|C18|C19|   C20|C21|
+--------------------+-----+--------+----+----------+--------+-----------+-------------+--------+----------+------------+---------+---------+------------+-----------+----------------+-----+---+---+----+---+---+------+---+
| 1000009418151094273|    0|14102100|1005|         0|1fbe01fe|   f3845767|     28905ebd|ecad2386|  7801e8d9|    07d7df22| a99f214a| ddd2926e|    44956a24|          1|               2|15706|320| 50|1722|  0| 35|    -1| 79|
|10000169349117863715|    0|14102100|1005|         0|1fbe01fe|   f3845767|     28905ebd|ecad2386|  7801e8d9|    

In [2]:
from pyspark.sql.functions import col

# drop duplicates
df = df.dropDuplicates()

# fill missing values
df = df.fillna("unknown")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
df.printSchema()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- id: string (nullable = false)
 |-- click: string (nullable = false)
 |-- hour: string (nullable = false)
 |-- C1: string (nullable = false)
 |-- banner_pos: string (nullable = false)
 |-- site_id: string (nullable = false)
 |-- site_domain: string (nullable = false)
 |-- site_category: string (nullable = false)
 |-- app_id: string (nullable = false)
 |-- app_domain: string (nullable = false)
 |-- app_category: string (nullable = false)
 |-- device_id: string (nullable = false)
 |-- device_ip: string (nullable = false)
 |-- device_model: string (nullable = false)
 |-- device_type: string (nullable = false)
 |-- device_conn_type: string (nullable = false)
 |-- C14: string (nullable = false)
 |-- C15: string (nullable = false)
 |-- C16: string (nullable = false)
 |-- C17: string (nullable = false)
 |-- C18: string (nullable = false)
 |-- C19: string (nullable = false)
 |-- C20: string (nullable = false)
 |-- C21: string (nullable = false)

In [5]:
# extract 1M to try CatBoost 
from pyspark.sql import functions as F
total_cnt = df.count()
target_total = 1_000_000
frac = target_total / total_cnt
print("Total rows:", total_cnt, "  frac:", frac)

fractions = {"0": frac, "1": frac}

df_sample = df.sampleBy(
    col="click",
    fractions=fractions,
    seed=2025
)

print("Sampled rows:", df_sample.count())

# label to int
df_sample = df_sample.withColumn("click", F.col("click").cast("int"))

# 3. write parquet to S3
out_path = "s3://aiml-final/catboost/catboost_sample_1M"
df_sample.write.mode("overwrite").parquet(out_path)

print("Saved to:", out_path)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Total rows: 31991090   frac: 0.03125870359528231
Sampled rows: 999856
Saved to: s3://aiml-final/catboost/catboost_sample_1M

In [4]:
# count unique values in each categories
from pyspark.sql.functions import approx_count_distinct

cols = [c for c in df.columns if c not in ["id", "click"]]

cardinality = df.agg(
    *[approx_count_distinct(c).alias(c) for c in cols]
)

cardinality.show(truncate=False)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----+---+----------+-------+-----------+-------------+------+----------+------------+---------+---------+------------+-----------+----------------+----+---+---+---+---+---+---+---+
|hour|C1 |banner_pos|site_id|site_domain|site_category|app_id|app_domain|app_category|device_id|device_ip|device_model|device_type|device_conn_type|C14 |C15|C16|C17|C18|C19|C20|C21|
+----+---+----------+-------+-----------+-------------+------+----------+------------+---------+---------+------------+-----------+----------------+----+---+---+---+---+---+---+---+
|217 |7  |7         |4256   |7676       |27           |8625  |521       |33          |2362471  |5174778  |7841        |5          |4               |2598|8  |9  |408|4  |63 |172|55 |
+----+---+----------+-------+-----------+-------------+------+----------+------------+---------+---------+------------+-----------+----------------+----+---+---+---+---+---+---+---+

In [5]:
df.count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

31991090

In [21]:
# check overall CTR
from pyspark.sql import functions as F

total_cnt = df.count()
pos_cnt = df.filter(F.col("click") == "1").count()
neg_cnt = total_cnt - pos_cnt

print("Total rows :", total_cnt)
print("Pos (click=1):", pos_cnt)
print("Neg (click=0):", neg_cnt)
print("CTR overall :", pos_cnt / total_cnt)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Total rows : 31991090
Pos (click=1): 5434977
Neg (click=0): 26556113
CTR overall : 0.16989033509017668

In [22]:
# check rows & CTR on a daily basis
df_day = df.withColumn("day", F.col("hour").substr(1, 6))

day_stats = (
    df_day
    .groupBy("day")
    .agg(
        F.count("*").alias("n_rows"),
        F.sum(F.col("click").cast("int")).alias("n_clicks")
    )
    .withColumn("ctr", F.col("n_clicks") / F.col("n_rows"))
    .orderBy("day")
)

day_stats.show(20, truncate=False)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------+-------+--------+-------------------+
|day   |n_rows |n_clicks|ctr                |
+------+-------+--------+-------------------+
|141021|3643416|634622  |0.17418323902623253|
|141022|4714508|741198  |0.1572164051901068 |
|141023|3419381|623174  |0.1822476056338852 |
|141024|2946054|515103  |0.17484506393976484|
|141025|2971666|541977  |0.18238153278329394|
|141026|3389326|620484  |0.18307002631201602|
|141027|2848083|517378  |0.18165832947986418|
|141028|4672604|711059  |0.15217617414187035|
|141029|3386052|529982  |0.15651915564202795|
+------+-------+--------+-------------------+

In [23]:
# check cum rows by day
from pyspark.sql.window import Window

w = Window.orderBy("day")

day_stats_cum = (
    day_stats
    .withColumn("cum_rows", F.sum("n_rows").over(w))
    .orderBy("day")
)

day_stats_cum.show(20, truncate=False)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------+-------+--------+-------------------+--------+
|day   |n_rows |n_clicks|ctr                |cum_rows|
+------+-------+--------+-------------------+--------+
|141021|3643416|634622  |0.17418323902623253|3643416 |
|141022|4714508|741198  |0.1572164051901068 |8357924 |
|141023|3419381|623174  |0.1822476056338852 |11777305|
|141024|2946054|515103  |0.17484506393976484|14723359|
|141025|2971666|541977  |0.18238153278329394|17695025|
|141026|3389326|620484  |0.18307002631201602|21084351|
|141027|2848083|517378  |0.18165832947986418|23932434|
|141028|4672604|711059  |0.15217617414187035|28605038|
|141029|3386052|529982  |0.15651915564202795|31991090|
+------+-------+--------+-------------------+--------+

In [6]:
# Take 3M rows
from pyspark.sql import functions as F

total_cnt = df.count()
print("Total rows:", total_cnt)

target_total = 3_000_000
frac = target_total / total_cnt
print("Target frac per class:", frac)

df = df.withColumn("day", F.col("hour").substr(1, 6))

fractions = {
    "0": frac,
    "1": frac
}

df_sample = df.sampleBy(
    col="click",
    fractions=fractions,
    seed=42
)

sample_cnt = df_sample.count()
print("Sampled rows:", sample_cnt)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Total rows: 31991090
Target frac per class: 0.09377611078584694
Sampled rows: 3000622

In [4]:
# Overall CTR check
total_cnt_s = df_sample.count()
pos_cnt_s = df_sample.filter(F.col("click") == "1").count()
neg_cnt_s = total_cnt_s - pos_cnt_s
print("Sample total rows:", total_cnt_s)
print("Sample Pos(click=1):", pos_cnt_s)
print("Sample Neg(click=0):", neg_cnt_s)
print("Sample CTR overall:", pos_cnt_s / total_cnt_s)

# By day CTR check
day_stats_sample = (
    df_sample
    .groupBy("day")
    .agg(
        F.count("*").alias("n_rows"),
        F.sum(F.col("click").cast("int")).alias("n_clicks")
    )
    .withColumn("ctr", F.col("n_clicks") / F.col("n_rows"))
    .orderBy("day")
)

day_stats_sample.show(20, truncate=False)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Sample total rows: 1000144
Sample Pos(click=1): 169932
Sample Neg(click=0): 830212
Sample CTR overall: 0.1699075333152026
+------+------+--------+-------------------+
|day   |n_rows|n_clicks|ctr                |
+------+------+--------+-------------------+
|141021|113832|19805   |0.17398446833930706|
|141022|146964|23082   |0.15705887156038215|
|141023|106914|19186   |0.1794526441813046 |
|141024|92064 |16151   |0.17543230795968023|
|141025|92989 |17155   |0.18448418630160557|
|141026|106048|19526   |0.1841241701870851 |
|141027|88972 |16372   |0.18401294789371936|
|141028|146193|22386   |0.15312634667870556|
|141029|106168|16534   |0.15573430788938286|
+------+------+--------+-------------------+

In [7]:
# Train/Val Split
df_pos = df_sample.filter(F.col("click") == "1")
df_neg = df_sample.filter(F.col("click") == "0")

print("Sample pos rows:", df_pos.count())
print("Sample neg rows:", df_neg.count())

# 80/20 random split
train_pos, val_pos = df_pos.randomSplit([0.8, 0.2], seed=42)
train_neg, val_neg = df_neg.randomSplit([0.8, 0.2], seed=42)

# to final train / val
train_df = train_pos.union(train_neg)
val_df   = val_pos.union(val_neg)

print("Train rows:", train_df.count())
print("Val rows:",   val_df.count())


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Sample pos rows: 510234
Sample neg rows: 2489613
Train rows: 2401443
Val rows: 599348

In [6]:
# Train CTR check
total_train = train_df.count()
pos_train = train_df.filter(F.col("click") == "1").count()
ctr_train = pos_train / total_train

print("=== Train CTR Check ===")
print("Train rows :", total_train)
print("Train pos  :", pos_train)
print("Train neg  :", total_train - pos_train)
print("Train CTR  :", ctr_train)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

=== Train CTR Check ===
Train rows : 799901
Train pos  : 136278
Train neg  : 663623
Train CTR  : 0.17036858311216013

In [7]:
# Val CTR check
total_val = val_df.count()
pos_val = val_df.filter(F.col("click") == "1").count()
ctr_val = pos_val / total_val

print("\n=== Val CTR Check ===")
print("Val rows :", total_val)
print("Val pos  :", pos_val)
print("Val neg  :", total_val - pos_val)
print("Val CTR  :", ctr_val)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…


=== Val CTR Check ===
Val rows : 199852
Val pos  : 34001
Val neg  : 165851
Val CTR  : 0.17013089686367913

In [8]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, FeatureHasher, VectorAssembler
from pyspark.ml import Pipeline

# low/medium cardinality cols - one-hot
low_mid_cols = [
    "banner_pos", "C1", "C15", "C16", "C18", "device_conn_type",
    "C17", "C19", "C21", "site_category", "app_domain"
]

# high cardinality cols- hash
hash_cols = ["device_id", "device_ip", "app_id", "device_model", "site_id", "site_domain"]

# StringIndexer
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_idx",
        handleInvalid="keep"
    )
    for c in low_mid_cols
]

# OneHotEncoder
encoder = OneHotEncoder(
    inputCols=[c + "_idx" for c in low_mid_cols],
    outputCols=[c + "_oh"  for c in low_mid_cols]
)

# Hasher
hasher = FeatureHasher(
    inputCols=hash_cols,
    outputCol="hash_features",
    numFeatures=2**16
)

# 4) VectorAssembler
assembler = VectorAssembler(
    inputCols=[c + "_oh" for c in low_mid_cols] + ["hash_features"],
    outputCol="features"
)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [9]:
pipeline = Pipeline(stages=indexers + [encoder, hasher, assembler])

# only fit on the train_df to avoid data leakage
pipeline_model = pipeline.fit(train_df)

# transform train / val
train_transformed = pipeline_model.transform(train_df)
val_transformed   = pipeline_model.transform(val_df)

#schema
train_transformed.select("click", "features").show(3, truncate=False)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----+----------------------------------------------------------------------------------------------------------------------------------------------------+
|click|features                                                                                                                                            |
+-----+----------------------------------------------------------------------------------------------------------------------------------------------------+
|1    |(66378,[0,7,14,22,31,35,56,439,506,561,581,1174,1927,6178,21829,45621,54974],[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]) |
|1    |(66378,[0,7,14,22,31,35,113,439,506,561,581,1174,1927,4895,21829,52163,54974],[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])|
|1    |(66378,[0,7,14,22,31,35,39,439,505,561,581,1174,1913,1927,21829,32498,54974],[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]) |
+-----+---------------------------------------------------

In [10]:
# only keep label + features
train_final = train_transformed.select("click", "features")
val_final   = val_transformed.select("click", "features")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [11]:
# change "click" to label

from pyspark.sql import functions as F
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

train_lr = train_final.withColumn("label", F.col("click").cast("double")).drop("click")
val_lr   = val_final.withColumn("label", F.col("click").cast("double")).drop("click")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [12]:
# Logistic Regression
from pyspark.sql.types import DoubleType
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.classification import LogisticRegression


get_p1 = F.udf(lambda v: float(v[1]), DoubleType())

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    rawPredictionCol="rawPrediction",
    probabilityCol="probability",
    maxIter=80,
    regParam=1e-4,
    elasticNetParam=0.0
)

lr_model = lr.fit(train_lr)

val_pred = (
    lr_model.transform(val_lr)
            .withColumn("p", get_p1(F.col("probability")))
)

val_pred.select("label", "probability", "p", "prediction").show(5, truncate=False)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----+----------------------------------------+-------------------+----------+
|label|probability                             |p                  |prediction|
+-----+----------------------------------------+-------------------+----------+
|1.0  |[0.8136166544965463,0.1863833455034537] |0.1863833455034537 |0.0       |
|1.0  |[0.8483285981386708,0.1516714018613292] |0.1516714018613292 |0.0       |
|1.0  |[0.6080636639463028,0.39193633605369715]|0.39193633605369715|0.0       |
|1.0  |[0.8606243056008468,0.1393756943991532] |0.1393756943991532 |0.0       |
|1.0  |[0.5440461016901027,0.4559538983098973] |0.4559538983098973 |0.0       |
+-----+----------------------------------------+-------------------+----------+
only showing top 5 rows

In [13]:
# LogLoss/AUC Check with different reg
from pyspark.ml.classification import LogisticRegression
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml.evaluation import BinaryClassificationEvaluator

eps = 1e-15
def compute_logloss(val_pred):
    p_clip = (
        F.when(F.col("p") < eps, eps)
         .when(F.col("p") > 1 - eps, 1 - eps)
         .otherwise(F.col("p"))
    )
    logloss_expr = -(
        F.col("label") * F.log(p_clip) +
        (1 - F.col("label")) * F.log(1 - p_clip)
    )
    return (
        val_pred
        .select(F.mean(logloss_expr).alias("logloss"))
        .collect()[0]["logloss"]
    )

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

reg_grid = [1e-5, 1e-4, 5e-4, 1e-3]
results = []

for reg in reg_grid:
    print(f"\n=== Training LR with regParam = {reg} ===")

    lr = LogisticRegression(
        featuresCol="features",
        labelCol="label",
        predictionCol="prediction",
        rawPredictionCol="rawPrediction",
        probabilityCol="probability",
        maxIter=80,
        regParam=reg,
        elasticNetParam=0.0
    )

    model = lr.fit(train_lr)

    val_pred = model.transform(val_lr).withColumn("p", get_p1(F.col("probability")))

    logloss = compute_logloss(val_pred)
    auc     = evaluator.evaluate(val_pred)

    print("Validation LogLoss:", logloss)
    print("Validation AUC    :", auc)

    results.append((reg, logloss, auc))

print("\nSummary:")
for reg, ll, auc in results:
    print(f"regParam={reg} -> logloss={ll:.6f}, AUC={auc:.6f}")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…


=== Training LR with regParam = 1e-05 ===
Validation LogLoss: 0.41424054388712633
Validation AUC    : 0.7271113776678508

=== Training LR with regParam = 0.0001 ===
Validation LogLoss: 0.4135842929181947
Validation AUC    : 0.7245364232151276

=== Training LR with regParam = 0.0005 ===
Validation LogLoss: 0.41329783564648925
Validation AUC    : 0.725000431916427

=== Training LR with regParam = 0.001 ===
Validation LogLoss: 0.4117174382776762
Validation AUC    : 0.7264042027508009

Summary:
regParam=1e-05 -> logloss=0.414241, AUC=0.727111
regParam=0.0001 -> logloss=0.413584, AUC=0.724536
regParam=0.0005 -> logloss=0.413298, AUC=0.725000
regParam=0.001 -> logloss=0.411717, AUC=0.726404

In [15]:
full_train = train_lr.unionByName(val_lr)

from pyspark.ml.classification import LogisticRegression
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType

def fit_lr_bootstrap(df, seed):
    lr = LogisticRegression(
        featuresCol="features",
        labelCol="label",
        maxIter=80,
        regParam=0.001,
        elasticNetParam=0.0
    )
    boot = df.sample(withReplacement=True, fraction=1.0, seed=seed)
    return lr.fit(boot)

model1 = fit_lr_bootstrap(full_train, seed=100)
model2 = fit_lr_bootstrap(full_train, seed=200)
model3 = fit_lr_bootstrap(full_train, seed=300)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [16]:
# read in test_df
test_df = spark.read.csv("s3://aiml-final/ProjectTestData.csv", header=True)
test_transformed = pipeline_model.transform(test_df)
test_lr = test_transformed.select("features")
test_lr.show(3, truncate=False)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----------------------------------------------------------------------------------------------------------------------------------------------------+
|features                                                                                                                                             |
+-----------------------------------------------------------------------------------------------------------------------------------------------------+
|(66378,[1,7,15,22,32,35,60,448,504,559,582,1366,11492,24769,39984,51140,57803],[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])|
|(66378,[0,11,14,22,32,36,482,559,774,17891,18127,39984,54787,56859,57803],[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])             |
|(66378,[0,7,14,22,31,35,439,505,560,581,1174,7358,19852,21829,26206,50723],[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])        |
+---------------------------------------------------------------------------------------

In [17]:
from pyspark.sql.functions import col
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType

get_p1 = F.udf(lambda v: float(v[1]), DoubleType())

# use model1 to predit
test_pred = (
    model1.transform(test_lr)
         .withColumn("p1", get_p1(col("probability")))
         .select("features", "p1")
)

# use model2 to predit
test_pred = (
    model2.transform(test_pred)
         .withColumn("p2", get_p1(col("probability")))
         .select("features", "p1", "p2")
)

# use model3 to predit
test_pred = (
    model3.transform(test_pred)
         .withColumn("p3", get_p1(col("probability")))
         .select("p1", "p2", "p3")
)

# ensemble：avg P
test_ens = test_pred.withColumn(
    "P(click)",
    (col("p1") + col("p2") + col("p3")) / 3.0
)

submission = test_ens.select("P(click)")
submission.show(5)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------------------+
|           P(click)|
+-------------------+
|0.21908610652709226|
|0.37392738862232666|
|0.03189757002840121|
|0.38841536486402006|
|0.35659723995102494|
+-------------------+
only showing top 5 rows

In [18]:
submission.coalesce(1).write.csv(
    "s3://aiml-final/submission_p_only/",
    header=True,
    mode="overwrite"
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [20]:
# RF Model
from pyspark.ml.classification import RandomForestClassifier
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType
get_p1 = udf(lambda v: float(v[1]), DoubleType())

# use only 50% to train RF
from pyspark.sql.functions import col

fractions = {
    0.0: 0.5,
    1.0: 0.5
}

train_lr_small = train_lr.stat.sampleBy(
    col="label",
    fractions=fractions,
    seed=42
)



rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    probabilityCol="probability",
    numTrees=20,
    maxDepth=5,
    maxBins=32,
    subsamplingRate=0.5,
    featureSubsetStrategy="sqrt",
    minInstancesPerNode=200,
    seed=42
)

rf_model = rf.fit(train_lr_small)

val_pred_rf = (
    rf_model.transform(val_lr)
             .withColumn("p", get_p1("probability"))
)

val_pred_rf.select("label", "p", "prediction").show(5, truncate=False)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----+-------------------+----------+
|label|p                  |prediction|
+-----+-------------------+----------+
|1.0  |0.17001074417201822|0.0       |
|1.0  |0.17001074417201822|0.0       |
|1.0  |0.17001074417201822|0.0       |
|1.0  |0.17001074417201822|0.0       |
|1.0  |0.17001074417201822|0.0       |
+-----+-------------------+----------+
only showing top 5 rows

In [21]:
# LogLoss
eps = 1e-15

p_clip = (
    F.when(F.col("p") < eps, eps)
     .when(F.col("p") > 1 - eps, 1 - eps)
     .otherwise(F.col("p"))
)

logloss_expr = -(
    F.col("label") * F.log(p_clip) +
    (1 - F.col("label")) * F.log(1 - p_clip)
)

logloss_val = (
    val_pred_rf
    .select(F.mean(logloss_expr).alias("logloss"))
    .collect()[0]["logloss"]
)

print("RF Validation LogLoss:", logloss_val)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

RF Validation LogLoss: 0.45507392948043246

In [22]:
# AUC
evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
auc_rf = evaluator.evaluate(val_pred_rf)
print("RF Validation AUC:", auc_rf)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

RF Validation AUC: 0.522985456994112

In [22]:
from pyspark.sql.functions import col

fractions = {
    0.0: 0.3,
    1.0: 0.3
}

train_lr_small = train_lr.stat.sampleBy(
    col="label",
    fractions=fractions,
    seed=42
)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [23]:
from pyspark.ml.classification import GBTClassifier

gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    maxIter=50,
    maxDepth=5,
    stepSize=0.1,
    subsamplingRate=0.8
)

gbt_model = gbt.fit(train_lr_small)

val_pred_gbt = (
    gbt_model
      .transform(val_lr)
      .withColumn("p", get_p1("probability"))
      .cache()
)

val_pred_gbt.select("label", "p", "prediction").show(5, truncate=False)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Interrupted by user


In [ ]:
logloss_gbt = (
    val_pred_gbt
      .select(F.mean(logloss_expr).alias("logloss"))
      .collect()[0]["logloss"]
)
print("GBT Validation LogLoss:", logloss_gbt)

auc_gbt = evaluator.evaluate(val_pred_gbt)
print("GBT Validation AUC:", auc_gbt)
